In [1]:
[
  {
    "prompt": "Explain what machine learning is in simple words.",
    "chosen": "Machine learning is a way to teach computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every case.",
    "rejected": "Machine learning is a machine that automatically learns everything without data."
  },
  {
    "prompt": "What is overfitting in machine learning?",
    "chosen": "Overfitting happens when a model memorizes the training data too closely and performs poorly on new unseen data.",
    "rejected": "Overfitting means the model becomes perfect for all types of data."
  },
  {
    "prompt": "Difference between classification and regression?",
    "chosen": "Classification predicts categories or labels, while regression predicts continuous numerical values.",
    "rejected": "Classification and regression are exactly the same thing."
  },
  {
    "prompt": "What is the purpose of a loss function?",
    "chosen": "A loss function measures how far a model’s predictions are from the true values, guiding the model during training.",
    "rejected": "A loss function increases model size and does not affect training."
  }
]

[{'prompt': 'Explain what machine learning is in simple words.',
  'chosen': 'Machine learning is a way to teach computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every case.',
  'rejected': 'Machine learning is a machine that automatically learns everything without data.'},
 {'prompt': 'What is overfitting in machine learning?',
  'chosen': 'Overfitting happens when a model memorizes the training data too closely and performs poorly on new unseen data.',
  'rejected': 'Overfitting means the model becomes perfect for all types of data.'},
 {'prompt': 'Difference between classification and regression?',
  'chosen': 'Classification predicts categories or labels, while regression predicts continuous numerical values.',
  'rejected': 'Classification and regression are exactly the same thing.'},
 {'prompt': 'What is the purpose of a loss function?',
  'chosen': 'A loss function measures how far a model’s predictions are from the 

In [3]:
!pip install torch transformers datasets trl peft accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 20.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [1]:
import os
import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import DPOTrainer, DPOConfig

In [4]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
# DATA_PATH = "data/preference_data.json"
OUTPUT_DIR = "outputs/dpo_model"

In [5]:
# with open(DATA_PATH, "r", encoding="utf-8") as f:
#     data = json.load(f)

# dataset = Dataset.from_list(data)

# print("Sample training row:")
# print(dataset[0])

In [6]:
from datasets import load_dataset
dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")

README.md:   0%|          | 0.00/643 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/131M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/2.14M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/62135 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [9]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"]
)

In [10]:
training_args = DPOConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_train_epochs=3,
    logging_steps=1,
    save_steps=20,
    save_total_limit=2,
    bf16=torch.cuda.is_available(),
    fp16=not torch.cuda.is_available(),
    gradient_checkpointing=True,
    report_to="none",
    max_length=512,
    beta=0.1,
    loss_type="sigmoid"
)

In [11]:
trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=peft_config
)

Extracting prompt from train dataset:   0%|          | 0/62135 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/62135 [00:00<?, ? examples/s]

In [12]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,0.693147
2,0.691703
3,0.698042
4,0.718176
5,0.671878
6,0.700904
7,0.689192
8,0.680883
9,0.684425
10,0.694165


KeyboardInterrupt: 

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Training completed. Model saved to: {OUTPUT_DIR}")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = "outputs/dpo_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

prompt = "Explain overfitting in machine learning."

messages = [
    {"role": "user", "content": prompt}
]

try:
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
except Exception:
    text = prompt

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)